# Optimize SCALP-lite Parameters

Download datasets first with notebook 00. This notebook defines parameters, runs PCA latent Bayesian optimization, with optional GPLVM refinement, and saves the best settings for downstream notebooks.


In [1]:
import os

# Set native thread/OpenMP controls before importing torch, sklearn, scanpy, or scalp_lite.
# This avoids macOS kernel crashes caused by mixed Intel/LLVM OpenMP runtimes.
for name in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[name] = "1"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("KMP_INIT_AT_FORK", "FALSE")

from scalp_lite.notebook_utils import (
    dataset_config,
    make_compact_search_space,
    make_estimator,
    make_scalp_optimization_objective,
    optimization_search_space,
    preprocess_params,
    run_latent_bayesopt,
    save_best_optimization_result,
)


Restart the kernel before running this notebook after changing OpenMP settings. The `notebooks/sitecustomize.py` file sets the same native-library controls at kernel startup, before ipykernel and compiled numerical libraries are imported.

In [2]:
selected_dataset = "scib_pancreas"
random_state = 0
embedding_method = "umap"
embedding_epochs = 60
invalid_score = -1e9
run_gplvm_refinement = True

pca_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 4,
    "embedding_model": "pca",
    "acquisition": "ei",
    "batch_size": 1,
}

gplvm_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 4,
    "embedding_model": "gplvm",
    "acquisition": "ei",
    "batch_size": 1,
}

dataset = dataset_config(selected_dataset)
raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
raw_adata = raw_estimator.to_data(dataset["input_path"])
raw_adata


AnnData object with n_obs × n_vars = 16382 × 19093
    obs: 'tech', 'celltype', 'size_factors'
    layers: 'counts'

In [3]:
fixed_preprocess_params = {"max_cells": 1000}

base_preprocess_params = preprocess_params(dataset, n_top_genes=1500)

base_estimator_params = {
    "n_components": 60,
}

base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}


In [4]:
preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 2000]},
}

estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 100]},
}

graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 35]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 6]},
    "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 20]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

search_space = optimization_search_space(
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
)
search_space


{'n_top_genes': {'type': 'int', 'bounds': [500, 2000]},
 'n_components': {'type': 'int', 'bounds': [20, 100]},
 'n_neighbors': {'type': 'int', 'bounds': [5, 35]},
 'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
 'n_inter_edges': {'type': 'int', 'bounds': [1, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.1, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [3, 20]},
 'rank_correction': {'type': 'categorical', 'values': [False, True]},
 'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}}

In [5]:
objective = make_scalp_optimization_objective(
    dataset=dataset,
    raw_adata=raw_adata,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
    embedding_method=embedding_method,
    embedding_epochs=embedding_epochs,
    invalid_score=invalid_score,
)


In [6]:
pca_result = run_latent_bayesopt(
    objective,
    search_space,
    **pca_bo_settings,
    random_state=random_state,
    invalid_score=invalid_score,
)

pca_result["best_params"], pca_result["best_score"]


failed params=preprocess={'n_top_genes': 1776, 'max_cells': 1000, 'min_cell_genes': None, 'min_gene_counts': 0, 'normalize': False, 'hvg_flavor': 'variance', 'create_artificial_batch': False}, estimator={'n_components': 71}, graph={'n_neighbors': 20, 'intra_fraction': 0.2286814667553363, 'n_inter_edges': 2, 'metric': 'euclidean', 'assignment_quantile': 0.1148748719756762, 'hubness_correction': 'csls', 'hubness_k': 6, 'rank_correction': True, 'edge_weighting': 'distance', 'mutual_neighbors': True, 'neighbor_mode': 'distance', 'symmetrize': True}: Missing required obs column: 'study'.
failed params=preprocess={'n_top_genes': 1255, 'max_cells': 1000, 'min_cell_genes': None, 'min_gene_counts': 0, 'normalize': False, 'hvg_flavor': 'variance', 'create_artificial_batch': False}, estimator={'n_components': 69}, graph={'n_neighbors': 35, 'intra_fraction': 0.580537494025796, 'n_inter_edges': 5, 'metric': 'euclidean', 'assignment_quantile': 0.9415651814089914, 'hubness_correction': 'csls', 'hubne

/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/botorch/models/utils/assorted.py:279: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)


failed params=preprocess={'n_top_genes': 1453, 'max_cells': 1000, 'min_cell_genes': None, 'min_gene_counts': 0, 'normalize': False, 'hvg_flavor': 'variance', 'create_artificial_batch': False}, estimator={'n_components': 100}, graph={'n_neighbors': 29, 'intra_fraction': 0.8994111459431287, 'n_inter_edges': 2, 'metric': 'euclidean', 'assignment_quantile': 0.8979148557505695, 'hubness_correction': 'csls', 'hubness_k': 15, 'rank_correction': True, 'edge_weighting': 'binary', 'mutual_neighbors': False, 'neighbor_mode': 'distance', 'symmetrize': True}: Missing required obs column: 'study'.


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/botorch/models/utils/assorted.py:279: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)


failed params=preprocess={'n_top_genes': 1152, 'max_cells': 1000, 'min_cell_genes': None, 'min_gene_counts': 0, 'normalize': False, 'hvg_flavor': 'variance', 'create_artificial_batch': False}, estimator={'n_components': 89}, graph={'n_neighbors': 35, 'intra_fraction': 0.8999999999999999, 'n_inter_edges': 5, 'metric': 'euclidean', 'assignment_quantile': 1.0, 'hubness_correction': 'csls', 'hubness_k': 16, 'rank_correction': True, 'edge_weighting': 'distance', 'mutual_neighbors': False, 'neighbor_mode': 'distance', 'symmetrize': True}: Missing required obs column: 'study'.


/Users/fabriziocosta/miniconda3/envs/py311/lib/python3.11/site-packages/botorch/models/utils/assorted.py:279: InputDataWarning: Data (outcome observations) is not standardized (std = tensor([0.], dtype=torch.float64), mean = tensor([0.], dtype=torch.float64)).Please consider scaling the input to zero mean and unit variance.
  check_standardization(Y=train_Y, raise_on_fail=raise_on_fail)


KeyboardInterrupt: 

In [ ]:
pca_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
compact_radii = {
    "n_top_genes": 300,
    "n_components": 20,
    "n_neighbors": 6,
    "intra_fraction": 0.1,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 4,
}

compact_search_space = make_compact_search_space(
    search_space,
    pca_result["best_params"],
    compact_radii,
    fix_categoricals=True,
)
compact_search_space


In [ ]:
if run_gplvm_refinement:
    gplvm_result = run_latent_bayesopt(
        objective,
        compact_search_space,
        **gplvm_bo_settings,
        random_state=random_state + 1,
        invalid_score=invalid_score,
    )
    gplvm_summary = (gplvm_result["best_params"], gplvm_result["best_score"])
else:
    gplvm_result = None
    gplvm_summary = "skipped"

gplvm_summary


In [ ]:
if gplvm_result is not None:
    gplvm_result["history"].sort_values("score", ascending=False).head(10)


In [ ]:
optimization_results = {"pca": pca_result}
if gplvm_result is not None:
    optimization_results["gplvm"] = gplvm_result

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = save_best_optimization_result(
    dataset_name=selected_dataset,
    optimization_results=optimization_results,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
)

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params
